# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing Croissant entities by their `@id`.

### Dataset Source
Provided as a Croissant schema at:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("Identifier:", metadata.identifier)
print("Authors' @id:", [a['@id'] for a in metadata.author])

## 2. Data Overview
Explore available record sets, fields, columns and their `@id`s referencing the Croissant schema structure.

*Note: The dataset includes three main tables, each represented as a record set:*
- Table 1: Clinical features
- Table 2: Hormone levels, radiological and ultrasound findings
- Table 3: Genetic variants

Let's enumerate their `@id`s and review the structure.

In [ ]:
# List available record sets and their @id (requires schema introspection)
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs['name']}")
    print("  Fields:")
    for f in rs['fields']:
        print(f"    - @id: {f['@id']}, name: {f['name']}, dataType: {f.get('dataType', 'N/A')}")
    print()

## 3. Data Extraction
Load data from each record set into pandas DataFrames. Entities are referenced via their Croissant `@id`.

For demonstration, we'll extract all three tables. Replace `<record_set_id>` with actual `@id` values from the schema overview above.

In [ ]:
# Collect record set @ids for each table
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set: {record_set_id}\nColumns: {df.columns.tolist()}\n")
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Process clinical features or hormone measurements by their Croissant field `@id`.

Here, we filter, normalize, and group by attributes for Table 2 (hormone levels).

In [ ]:
# Select Table 2 (Hormone and Imaging Data) by its @id
table2_id = None
for rs in record_sets:
    if 'Hormone' in rs['name'] or 'Table 2' in rs['name'] or 'Assay' in rs['name']:
        table2_id = rs['@id']
        break

df2 = dataframes.get(table2_id, pd.DataFrame())

if not df2.empty:
    # Find a numeric field (e.g., hormone_level @id)
    numeric_field_id = None
    for f in [f for f in rs['fields'] if rs['@id']==table2_id]:
        if f.get('dataType', '').lower() in ['float', 'integer', 'number']:
            numeric_field_id = f['@id']
            break

    if numeric_field_id is None and len(df2.select_dtypes('number').columns) > 0:
        numeric_field_id = df2.select_dtypes('number').columns[0]

    if numeric_field_id is not None:
        threshold = df2[numeric_field_id].mean() if df2[numeric_field_id].count() else 10
        filtered_df2 = df2[df2[numeric_field_id] > threshold]
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        print(filtered_df2.head())

        # Normalize the chosen numeric field
        filtered_df2[numeric_field_id + '_normalized'] = (
            filtered_df2[numeric_field_id] - filtered_df2[numeric_field_id].mean()
        ) / filtered_df2[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df2[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Try grouping by a categorical field
        group_field_id = None
        for f in [f for f in rs['fields'] if rs['@id']==table2_id]:
            if f.get('dataType', '').lower() == 'text':
                group_field_id = f['@id']
                break

        if group_field_id and group_field_id in filtered_df2.columns:
            grouped_df = filtered_df2.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped filtered records by '{group_field_id}':")
            print(grouped_df)
    else:
        print("No numeric field found in Table 2 record set.")
else:
    print("Table 2 (Hormone levels) not found or empty.")

## 5. Visualization
Visualize the distribution of hormone levels from Table 2 record set, again referencing fields by their `@id`.

If available, plot a histogram and a boxplot.

In [ ]:
# Visualize the numeric field selected in EDA
if 'filtered_df2' in locals() and not filtered_df2.empty and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    plt.hist(filtered_df2[numeric_field_id], bins=10, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(6,4))
    plt.boxplot(filtered_df2[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id} (filtered)")
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset package, extracted and referenced metadata and records by their Croissant `@id`, processed hormone level data, performed normalization and grouping, and visualized the results. Due to the small cohort size (N=3), the dataset is valuable for clinical insight and rare disease research, but results should not be generalized without further data. All steps used `mlcroissant`, promoting reproducibility and transparency.
